In [ ]:
!pip install asyncpraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 4.9 MB/s eta 0:00:00


In [ ]:
import asyncpraw
import asyncio
import pandas as pd
import re

# --- CONFIG ---
reddit_client_id = ""
reddit_client_secret = ""
reddit_user_agent = ""

# Neurodivergence-related terms
neuro_terms = ["autism", "autistic", "neurodivergent", "Asperger", "ADHD",
               "representation", "portrayal", "spectrum"]

# Context words that suggest movie discussion
context_words = ['film', 'movie', 'series', 'show', 'character', 'portrays',
                 'depicts', 'stars', 'performance', 'actor', 'actress',
                 'review', 'watch', 'streaming', 'episode', 'season']

# Movies list
movies = [
    {"title": "Atypical", "type": "tv", "year": 2017},
    {"title": "The Good Doctor", "type": "tv", "year": 2017},
    {"title": "Temple Grandin", "type": "film", "year": 2010},
    {"title": "Everything's Gonna Be Okay", "type": "tv", "year": 2020},
    {"title": "Taare Zameen Par", "type": "film", "year": 2007, "alt_titles": ["Like Stars on Earth"]},
    {"title": "Sitare Zameen Par", "type": "film", "year": 2025},
    {"title": "Front of the Class", "type": "film", "year": 2008},
    {"title": "The Tic Code", "type": "film", "year": 1998},
    {"title": "Patience", "type": "film", "year": 2017},
    {"title": "Barfi", "type": "film", "year": 2012},
    {"title": "My Name is Khan", "type": "film", "year": 2010, "alt_titles": ["MNIK"]},
    {"title": "Music", "type": "film", "year": 2021, "director": "Sia"},
    {"title": "Hichki", "type": "film", "year": 2018, "alt_titles": ["Hiccup"]},  # informal translation
    {"title": "Rain Man", "type": "film", "year": 1988},
    {"title": "Koi... Mil Gaya", "type": "film", "year": 2003, "alt_titles": ["Found Someone", "I Have Found Someone"]},
    {"title": "Extraordinary Attorney Woo", "type": "tv", "year": 2022, "alt_titles": ["Weird Lawyer Woo Young-woo"]}
]

# --- FUNCTIONS ---
def mentions_title_with_boundary(text: str, title: str, alt_titles: list = None) -> bool:
    lower_text = text.lower()
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        if re.search(pattern, lower_text):
            return True
    else:
        if title.lower() in lower_text:
            return True
    if alt_titles:
        for alt in alt_titles:
            if alt.lower() in lower_text:
                return True
    return False

def has_context_words(text: str) -> bool:
    lower_text = text.lower()
    return any(word in lower_text for word in context_words)

def count_mentions(text: str, title: str, alt_titles: list = None) -> int:
    lower_text = text.lower()
    count = 0
    if len(title.split()) <= 2:
        pattern = r'\b' + re.escape(title.lower()) + r'\b'
        count += len(re.findall(pattern, lower_text))
    else:
        start = 0
        while True:
            pos = lower_text.find(title.lower(), start)
            if pos == -1:
                break
            count += 1
            start = pos + 1
    if alt_titles:
        for alt in alt_titles:
            count += lower_text.count(alt.lower())
    return count

# --- MAIN ASYNC EXECUTION ---
async def collect_reddit_data():
    reddit = asyncpraw.Reddit(
        client_id=reddit_client_id,
        client_secret=reddit_client_secret,
        user_agent=reddit_user_agent
    )

    all_posts = []

    for movie in movies:
        title = movie["title"]
        alt_titles = movie.get("alt_titles", [])
        print(f"Searching Reddit for '{title}'...")

        subreddit = await reddit.subreddit("all")
        async for submission in subreddit.search(title, limit=200):
            combined_text = f"{submission.title} {submission.selftext}"
            if not mentions_title_with_boundary(combined_text, title, alt_titles):
                continue
            if not has_context_words(combined_text):
                continue

            mention_count = count_mentions(combined_text, title, alt_titles)

            # Collect comments
            await submission.load()
            await submission.comments.replace_more(limit=0)

            # Use normal for-loop on the list
            comments_list = submission.comments.list()  # this is a regular list
            comments_text = " ".join([c.body for c in comments_list])
            mention_count_comments = count_mentions(comments_text, title, alt_titles)

            all_posts.append({
                "movie_title": title,
                "movie_type": movie.get("type", "film"),
                "reddit_id": submission.id,
                "title": submission.title,
                "selftext": submission.selftext,
                "comments_text": comments_text,
                "mention_count": mention_count,
                "mention_count_comments": mention_count_comments,
                "url": submission.url,
                "score": submission.score,
                "num_comments": submission.num_comments,
                "subreddit": submission.subreddit.display_name
            })

    # Convert to DataFrame
    df = pd.DataFrame(all_posts)
    df_filtered = df[(df["mention_count"] >= 1) | (df["mention_count_comments"] >= 2)]

    # Save CSV
    df.to_csv("reddit_movie_posts_all.csv", index=False)
    df_filtered.to_csv("reddit_movie_posts_filtered.csv", index=False)

    print("Reddit async data collection complete!")
    print(f"Total posts found: {len(df)}")
    print(f"Filtered high-quality posts: {len(df_filtered)}")

# --- RUN ---
await collect_reddit_data()


Searching Reddit for 'Atypical'...


ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7c914779c2c0>


Searching Reddit for 'The Good Doctor'...
Searching Reddit for 'Temple Grandin'...
Searching Reddit for 'Everything's Gonna Be Okay'...
Searching Reddit for 'Taare Zameen Par'...
Searching Reddit for 'Sitare Zameen Par'...
Searching Reddit for 'Front of the Class'...
Searching Reddit for 'The Tic Code'...
Searching Reddit for 'Patience'...
Searching Reddit for 'Barfi'...
Searching Reddit for 'My Name is Khan'...
Searching Reddit for 'Music'...
Searching Reddit for 'Hichki'...
Searching Reddit for 'Rain Man'...
Searching Reddit for 'Koi... Mil Gaya'...
Searching Reddit for 'Extraordinary Attorney Woo'...
Reddit async data collection complete!
Total posts found: 595
Filtered high-quality posts: 595


In [ ]:
from google.colab import files

files.download("reddit_movie_posts_filtered.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>